# ✈️ AeroLex — Building a Production-Grade RAG Pipeline

A hands-on build of a multi-stage Retrieval-Augmented Generation system over
**airline passenger-rights law** (EU 261/2004, US DOT 14 CFR 260, Delta Contracts
of Carriage).

**The pipeline:**
```
question
  └─▶ condense follow-up            (conversational memory)       [#8]
  └─▶ BM25 lexical  +  Gemini dense  (hybrid retrieval)           [#2,#3,#4]
  └─▶ reciprocal rank fusion         (merge the two)              [#5]
  └─▶ FLAIR feedback re-scoring      (learn from 👍/👎)            [#10]
  └─▶ cross-encoder re-ranking       (precise top-k)              [#6]
  └─▶ grounded answer + [[n]] cites  (+ long-term profile)        [#7,#9]
```

**How to run:** pick the **`Python (aerolex RAG)`** kernel, then run top to bottom.
Everything works offline (extractive answers); with a `GEMINI_API_KEY` you get full
generation. You built each stage? The final section assembles them with the
`aerolex` library.


# Why and when do we use RAG?
# Best practices
# See if we can implement SOTA for our respective sections - if yes, great, if no, try to do a simpler version or talk about it

## 0 · Setup

In [1]:
import time
from pathlib import Path
import numpy as np
from aerolex import config
from aerolex.loaders import load_documents
from aerolex.chunking import chunk_corpus, load_chunks
from aerolex.store import InMemoryVectorStore
from aerolex.embeddings import GeminiEmbedder
from aerolex.bm25 import BM25Index

In [2]:
# Set API key, LLM, Embedding Model 
EMBED_MODEL = "gemini-embedding-001"
EMBED_DIM = 768
GEN_MODEL = "gemini-3.5-flash-lite"
KEY = config.load_api_key()
CACHE_DIR = r"C:\src\IEEEWorkshop\aerolex-production-rag\cache"

## 1 · Hierarchical Chunking  ·  Feature #1 - Advitya
Tasks:
- Figure out how to pull information from referenced sections/articles
- One idea - Add references key in the json and add chunk ids of referenced articles

Legal text is a **hierarchy** (Article / § / RULE → paragraph). We split on those
*logical* boundaries — never fixed-size windows — and attach citation metadata, so
an answer can cite *"EU 261/2004 — Article 7"* instead of *"chunk 42"*.

In [3]:
docs = load_documents()
for d in docs:
    print(f"{d.doc_id:16s} {d.doc_type:6s} {len(d.text):>7,d} chars  {d.title}")

# Prefer the pre-built cache (instant); otherwise chunk live.
try:
    chunks = load_chunks()
    print(f"\nloaded {len(chunks)} chunks from cache")
except FileNotFoundError:
    chunks = chunk_corpus(docs)
    print(f"\nchunked live: {len(chunks)} chunks")

by_id = {c.chunk_id: c for c in chunks}
art7 = next(c for c in chunks if c.section_label == "Article 7")
print("\nExample chunk —", art7.citation)
print(art7.text[:320], "…")

delta_domestic   delta   78,663 chars  Delta Domestic General Rules Tariff (Contract of Carriage)
delta_intl       delta  118,099 chars  Delta International General Rules Tariff (Contract of Carriage)
eu261            eu      27,544 chars  EU Regulation (EC) No 261/2004
cfr260           cfr     26,287 chars  US DOT 14 CFR Part 260

loaded 234 chunks from cache

Example chunk — EU Regulation (EC) No 261/2004 — Article 7
1.
Where reference is made to this Article, passengers shall
receive compensation amounting to:
(a) EUR 250 for all flights of 1 500 kilometres or less;
(b) EUR 400 for all intra-Community flights of more than
1 500 kilometres, and for all other flights between 1 500
and 3 500 kilometres;
(c) EUR 600 for all flights no …


## Emebdding Creation, Chunking eval - Nutan

In [4]:
print(f"{len(chunks)} chunks loaded!")

234 chunks loaded!


In [28]:
if not KEY:
    print("GEMINI_API_KEY not set.")

print(f"Embedding {len(chunks)} chunks with {EMBED_MODEL} "
        f"(dim={EMBED_DIM}) …")

t0 = time.time()
embedder = GeminiEmbedder(api_key=KEY, model=EMBED_MODEL, dim=EMBED_DIM)
matrix = embedder.embed_documents([c.searchable_text for c in chunks])
print(f"Embedded in {time.time() - t0:.1f}s, shape={matrix.shape}")

Embedding 234 chunks with gemini-embedding-001 (dim=768) …
Embedded in 132.4s, shape=(234, 768)


In [ ]:
# create a matrix with id and store using numpy

store = InMemoryVectorStore([c.chunk_id for c in chunks], matrix)
store.save(Path(CACHE_DIR) / "new_embeddings.npz")

In [5]:
# if time out errors load, already created vector store
store = InMemoryVectorStore.load(Path(CACHE_DIR) / "new_embeddings.npz")
print (f"{store.matrix.shape[0]} Embeddings loaded successfully of shape {store.matrix.shape[1]}!")

234 Embeddings loaded successfully of shape 768!


## 2 · Lexical Search with BM25  ·  Feature #2 - Nutan

BM25 (Best Match 25) is a probabilistic ranking algorithm to measure how relevant a document is to a query, combining term frequency, inverse document frequency, and document‑length normalization.
Use `BM25Okapi` directly with the same stopword-filtering tokenizer for chunks and queries. Scores are lexical relevance scores, not cosine similarities.

### Keyword tokenization

In [6]:
import re
from sklearn.feature_extraction.text import ENGLISH_STOP_WORDS

def tokenize(text: str) -> list[str]:
    # TODO Fetch list of keywords 
    tokens = re.findall(r"[a-z]+|\u00A7|\d+(?:[./-]\d+)*", text.lower())
    return [token for token in tokens if token not in ENGLISH_STOP_WORDS]

assert tokenize("What is the refund for a cancelled flight?") == ["refund", "cancelled", "flight"]
print("tokenize OK")

tokenize OK


In [7]:
# Tokenize the query for BM25 search
query_tokens = tokenize("What is the refund for a cancelled flight")

In [8]:
# Initialize BM25 index with chunk IDs and their corresponding searchable text
bm25 = BM25Index([c.chunk_id for c in chunks], [c.searchable_text for c in chunks])

In [30]:
kw_search_chunks = []
for cid, score in bm25.search(query_tokens, top_k=5):
    print(f"{score:6.2f}  {by_id[cid].citation}")
    kw_search_chunks.append((cid, score))

  5.21  US DOT 14 CFR Part 260 — § 260.6 (part 4/6)
  4.72  US DOT 14 CFR Part 260 — § 260.1
  4.67  US DOT 14 CFR Part 260 — § 260.6 (part 6/6)
  4.66  US DOT 14 CFR Part 260 — § 260.6 (part 2/6)
  4.47  US DOT 14 CFR Part 260 — § 260.6 (part 5/6)


## 3 · Vector Search - Nutan

Gemini embeds every chunk into a vector; the whole corpus is saved in one NumPy matrix.
Similarity is just a **normalized dot product** (cosine) — no vector database.

### 🛠️ TODO: write `cosine_search` (normalized dot product top-k)

In [12]:
def cosine_search(query_vec, matrix, ids, top_k=5):

    # TODO: 1) L2-normalize query_vec and each row of matrix
    q_hat = query_vec / np.sqrt(np.sum(query_vec ** 2))
    matrix_hat = matrix / np.sqrt(np.sum(matrix ** 2, axis=1, keepdims=True))
    #       2) scores = matrix @ query   (cosine, since normalized)
    scores = matrix_hat @ q_hat
    #       3) return the top_k (id, score) pairs, highest first
    top_k = np.argsort(-scores)[:top_k]
    top_chunks = [(ids[i], scores[i]) for i in top_k]
    return top_chunks

In [13]:
# def cosine_search(query_vec, matrix, ids, top_k=5):
    # TODO: 1) L2-normalize query_vec and each row of matrix
    #       2) scores = matrix @ query   (cosine, since normalized)
    #       3) return the top_k (id, score) pairs, highest first
    # raise NotImplementedError("your turn!")

_hit = cosine_search(store.matrix[0], store.matrix, store.ids, top_k=5)[0]
assert _hit[0] == store.ids[0] and abs(_hit[1] - 1.0) < 1e-4
print("cosine_search OK")

cosine_search OK


In [26]:
# Embed a real query (1 API call) and search. Needs a key; else we reuse a chunk vector.
query = "What is the refund for a cancelled flight?"

query_vector = GeminiEmbedder(api_key=KEY, model=EMBED_MODEL, dim=EMBED_DIM).embed_query(query)


dense_hits = cosine_search(query_vector, store.matrix, store.ids, top_k=5)
vector_search_chunks = []
for cid, s in dense_hits:
    vector_search_chunks.append((cid, s))
    print(f"{s:5.3f}  {by_id[cid].citation}")

0.720  Delta International General Rules Tariff (Contract of Carriage) — RULE 23 (part 1/7)
0.719  Delta Domestic General Rules Tariff (Contract of Carriage) — RULE 22 (part 1/6)
0.707  US DOT 14 CFR Part 260 — § 260.6 (part 2/6)
0.696  Delta Domestic General Rules Tariff (Contract of Carriage) — RULE 22 (part 2/6)
0.694  US DOT 14 CFR Part 260 — § 260.4 (part 1/3)


## 4 · Reciprocal Rank Fusion  ·  Feature #5

BM25 scores and cosine similarity scores live on different scales and produce two different sets of ranked documents. So we fuse by **rank**:
each list contributes `1 / (k + rank)` to an id's score (`k = 60`).

### 🛠️ TODO: write `rrf`

In [28]:
vector_search_chunks

[('delta_intl::rule-23::0', np.float32(0.7197001)),
 ('delta_domestic::rule-22::0', np.float32(0.7194926)),
 ('cfr260::260-6::1', np.float32(0.70701617)),
 ('delta_domestic::rule-22::1', np.float32(0.69601727)),
 ('cfr260::260-4::0', np.float32(0.6936874))]

In [31]:
kw_search_chunks

[('cfr260::260-6::3', 5.209017420297043),
 ('cfr260::260-1::0', 4.716110541876705),
 ('cfr260::260-6::5', 4.665165771246034),
 ('cfr260::260-6::1', 4.655950412277342),
 ('cfr260::260-6::4', 4.467967284438473)]

In [68]:
def rrf(vector_search_chunks, kw_search_chunks, k=60):
    # TODO 
    combined_chunks = [chunk[0] for chunk in kw_search_chunks] + [chunk[0] for chunk in vector_search_chunks]
    from collections import defaultdict
    kw_chunk_rank = defaultdict(int, {chunk[0]: index + 1 for index, chunk in enumerate(kw_search_chunks)})
    vector_chunk_rank = defaultdict(int, {chunk[0]: index + 1 for index, chunk in enumerate(vector_search_chunks)})
    rrf_scores = {}
    for chunk in combined_chunks:
        r1 = kw_chunk_rank[chunk] # rank in bm25
        r2 = vector_chunk_rank[chunk] # rank in vector search
        bm25_term = 1/(r1 + k) if r1 > 0 else 0
        vector_term = 1/(r2 + k) if r2 > 0 else 0
        final_rrf_score = bm25_term + vector_term
        rrf_scores[chunk] = final_rrf_score
    ranked_chunks = sorted(
    rrf_scores.items(), key=lambda item: item[1], reverse=True
)
    return ranked_chunks

In [69]:
ranked_chunks = rrf(kw_search_chunks, vector_search_chunks, k=60)

In [70]:
ranked_chunks

[('cfr260::260-6::1', 0.03149801587301587),
 ('delta_intl::rule-23::0', 0.01639344262295082),
 ('cfr260::260-6::3', 0.01639344262295082),
 ('delta_domestic::rule-22::0', 0.016129032258064516),
 ('cfr260::260-1::0', 0.016129032258064516),
 ('cfr260::260-6::5', 0.015873015873015872),
 ('delta_domestic::rule-22::1', 0.015625),
 ('cfr260::260-4::0', 0.015384615384615385),
 ('cfr260::260-6::4', 0.015384615384615385)]

In [ ]:
# More efficient way
# def rrf(rankings, k=60, top_k=10):
#     scores = {}
#     for ranking in rankings:
#         for rank, chunk_id in enumerate(ranking, start=1):
#             scores[chunk_id] = scores.get(chunk_id, 0.0) + 1.0 / (k + rank)
#     return sorted(scores.items(), key=lambda item: item[1], reverse=True)[:top_k]

# assert rrf([["b", "a", "c"], ["b", "a", "d"]])[0][0] == "b"
# print("rrf OK")

rrf OK


In [ ]:
# bm25_scores = bm25.get_scores(tokenize(query))
# top_indices = sorted(range(len(chunks)), key=lambda i: -bm25_scores[i])[:20]
# bm25_ids = [chunks[i].chunk_id for i in top_indices]
# dense_ids = [cid for cid, _ in dense_hits]
# fused = rrf([bm25_ids, dense_ids], k=config.RRF_K, top_k=10)
# print("Fused top-5:")
# for cid, s in fused[:5]:
#     print(f"{s:.4f}  {by_id[cid].citation}")

## 5 · Cross-Encoder Re-Ranking  ·  Feature #6 - TBD

First-stage retrieval scores query and document *independently*. A cross-encoder
reads them **together** (deep cross-attention) and is far more precise — so we run
it only over the small fused pool. (Model is pre-cached; runs locally.)

In [ ]:
from aerolex.rerank import CrossEncoderReranker

candidates = [(cid, by_id[cid].searchable_text) for cid, _ in fused]
reranked = CrossEncoderReranker().rerank(query, candidates, top_k=config.RERANK_TOP_K)
print("Re-ranked top chunks:")
for cid, s in reranked:
    print(f"{s:7.2f}  {by_id[cid].citation}")
final_chunks = [by_id[cid] for cid, _ in reranked]

## 6 · Grounded Answer with Inline Citations  ·  Feature #7 - Advitya

- What is the state of the art in citations?
- See if we can implement it 

The model answers **only** from numbered passages and tags each claim with `[[n]]`.
A regex post-processor then validates those tags against the real passages and
**drops hallucinated ones** — the difference between "looks cited" and "is cited".

### 🛠️ TODO: write `resolve_citations`

In [ ]:
CITE = re.compile(r"\[\[(\d+)\]\]")

def resolve_citations(answer, index):
    # TODO: keep only [[n]] whose n is a key in `index`; delete the rest.
    # Return (cleaned_answer, ["[[n]] <citation>", ...] for the n's actually used).
    raise NotImplementedError("your turn!")

_c, _r = resolve_citations("A [[1]] and B [[9]].", {1: "EU261 Art 7", 2: "EU261 Art 5"})
assert "[[9]]" not in _c and _r == ["[[1]] EU261 Art 7"]
print("resolve_citations OK")

In [ ]:
from aerolex.generate import Generator

result = Generator().generate(query, final_chunks)
print("OFFLINE" if result["offline"] else "GEMINI", "answer:\n")
print(result["answer"])
print("\nReferences:")
for r in result["references"]:
    print("  ", r)

## Evaluation Strategies - Nutan

## 10 · The Full Pipeline

Now assemble the notebook retrieval steps with the `aerolex` generation and memory
components, using `BM25Okapi` directly, and ask a question end to end.

In [ ]:
if key:
    from aerolex.embeddings import GeminiEmbedder

    question = "If my 4000 km flight is cancelled, what am I owed?"
    standalone = mem.condense(question)
    query_vec = GeminiEmbedder().embed_query(standalone)
    dense_results = cosine_search(query_vec, store.matrix, store.ids, top_k=config.DENSE_TOP_K)
    sparse_scores = bm25.get_scores(tokenize(standalone))
    sparse_order = sorted(range(len(chunks)), key=lambda i: -sparse_scores[i])[:config.BM25_TOP_K]
    fused_results = rrf([[chunks[i].chunk_id for i in sparse_order],
                         [cid for cid, _ in dense_results]],
                        k=config.RRF_K, top_k=config.RRF_TOP_K)
    candidates = [(cid, by_id[cid].searchable_text) for cid, _ in fused_results]
    ranked = CrossEncoderReranker().rerank(standalone, candidates, top_k=config.RERANK_TOP_K)
    answer_chunks = [by_id[cid] for cid, _ in ranked]
    out = Generator().generate(standalone, answer_chunks,
                               profile_block=prof.context_block("demo_user"))
    out.update(query=question, standalone_query=standalone, chunks=answer_chunks)
    mem.add_user(question)
    mem.add_assistant(out["answer"])
    print(out["answer"])
    print("\nReferences:")
    for r in out["references"]:
        print("  ", r)
else:
    print("Set GEMINI_API_KEY to run the full generative pipeline.")

## 11 · Where to go next — citations, state of the art

What we built is **Level 1** citation: the model self-tags `[[n]]`, and we validate
the *number is real*. State-of-the-art adds **verification**:

- **Level 2** structured output — model returns `{sentence, support_ids, quote}`.
- **Level 3** quote-grounded spans — match the quote back into the source for exact
  offsets; drop unverifiable quotes.
- **Level 4** NLI / entailment (ALCE) — check the passage actually *entails* the
  claim; score citation **precision & recall**.

The retrieval stack (chunking → hybrid → fusion → re-rank) is where most answer
quality is won. 🎓 You built all of it.